# DocuChat AI — Clean Colab QLoRA Fine-Tuning

This notebook fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` using 4-bit QLoRA on a free Colab T4 GPU. It avoids the earlier `torchvision::nms` and `bitsandbytes` version conflict.

Before starting: choose **Runtime → Change runtime type → T4 GPU**. Run one cell at a time.

## 1. Confirm that Colab assigned a GPU

In [1]:
!nvidia-smi
import sys
print("Python:", sys.version)

Fri Sep 18 14:27:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install one compatible package set

Do not run any older installation cell before or after this one. PyTorch is supplied by Colab and is deliberately not reinstalled.

In [2]:
%pip uninstall -y -q bitsandbytes transformers datasets peft trl accelerate
%pip install -q --upgrade-strategy only-if-needed \
  transformers==4.48.3 \
  datasets==3.2.0 \
  peft==0.14.0 \
  trl==0.15.1 \
  accelerate==1.3.0 \
  bitsandbytes==0.50.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does n

### Restart once

Now choose **Runtime → Restart session**, then continue from Step 3. Do not rerun the installation cell after restarting.

## 3. Verify the restarted environment

In [3]:
import torch, transformers, datasets, peft, trl, accelerate, bitsandbytes

assert torch.cuda.is_available(), "GPU not detected. Select a T4 GPU runtime."
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
GPU: Tesla T4
Transformers: 4.48.3
Datasets: 3.2.0
PEFT: 0.14.0
TRL: 0.15.1
Accelerate: 1.3.0
bitsandbytes: 0.50.2


## 4. Download your GitHub repository

In [4]:
from pathlib import Path
import os

REPO_URL = "https://github.com/karangy7904/DocuChat-AI-Fine-Tuned-PDF-Question-Answering-Assistant.git"
REPO_DIR = Path("/content/DocuChat-AI")

if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("Repository already exists; using the existing folder.")

assert REPO_DIR.exists(), "Repository download failed. Check that the GitHub repository is public."
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

Working directory: /content/DocuChat-AI


## 5. Select and validate the training dataset

In [5]:
import json
from pathlib import Path

preferred = Path("data/reviewed_training_data.jsonl")
fallback = Path("data/training_data.jsonl")
DATASET_PATH = preferred if preferred.exists() else fallback
assert DATASET_PATH.exists(), f"Training file not found: {DATASET_PATH}"

records = []
for line_number, line in enumerate(DATASET_PATH.read_text(encoding="utf-8").splitlines(), 1):
    if not line.strip():
        continue
    row = json.loads(line)
    missing = {"instruction", "context", "response"} - row.keys()
    assert not missing, f"Line {line_number} is missing: {sorted(missing)}"
    assert all(str(row[k]).strip() for k in ("instruction", "context", "response")), f"Empty value on line {line_number}"
    records.append(row)

assert records, "The training dataset is empty."
print("Dataset:", DATASET_PATH)
print("Valid examples:", len(records))
print("First example:", records[0])

Dataset: data/training_data.jsonl
Valid examples: 5
First example: {'instruction': 'What is machine learning?', 'context': 'Machine learning enables computer systems to learn patterns from data.', 'response': 'Machine learning enables computer systems to learn patterns from data.'}


## 6. Fine-tune with 4-bit QLoRA

This uses small T4-safe batches. The base model is public, so no Hugging Face token is required.

In [6]:
from pathlib import Path
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "models/docuchat-lora-final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map="auto",
    use_cache=False,
)

train_dataset = load_dataset("json", data_files=str(DATASET_PATH), split="train")

def format_record(record):
    messages = [
        {"role": "system", "content": "Answer only from the supplied document context. If the answer is absent, say: I could not find this information in the supplied document."},
        {"role": "user", "content": f"Context:\n{record['context']}\n\nQuestion:\n{record['instruction']}"},
        {"role": "assistant", "content": record["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_seq_length=512,
    logging_steps=5,
    save_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    ),
    formatting_func=format_record,
    processing_class=tokenizer,
)

trainer.train()
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to:", OUTPUT_DIR)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

...

Applying chat template to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Step,Training Loss


Saved LoRA adapter to: models/docuchat-lora-final


## 7. Test the fine-tuned adapter

In [7]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

del trainer, model
torch.cuda.empty_cache()

test_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
test_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

context = "Plivo products handle over 1 billion API requests per month."
question = "How many API requests does Plivo handle per month?"
messages = [
    {"role": "system", "content": "Answer only from the supplied document context."},
    {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"},
]
prompt = test_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = test_tokenizer(prompt, return_tensors="pt").to(test_model.device)
with torch.inference_mode():
    output = test_model.generate(**inputs, max_new_tokens=64, do_sample=False)
new_tokens = output[0, inputs.input_ids.shape[1]:]
answer = test_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print("Fine-tuned answer:", answer)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Fine-tuned answer: According to the provided context, Plivo handles approximately **1 billion** API requests per month.


## 8. Download the final adapter

In [8]:
import shutil
from google.colab import files

archive = shutil.make_archive("docuchat-lora-final", "zip", OUTPUT_DIR)
print("Created:", archive)
files.download(archive)

Created: /content/DocuChat-AI/docuchat-lora-final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Important notes

- Keep `adapter_model.safetensors` out of normal Git commits if GitHub rejects it for exceeding 100 MB; use Git LFS when necessary.
- Do not commit Hugging Face tokens or other secrets. This public base model does not require a token.
- Fine-tuning changes answer style and behavior; FAISS retrieval still supplies the relevant PDF text.
- For a meaningful evaluation, keep several question-answer pairs out of training and test them separately.